# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 3 we turned each card's history into a flat token stream. Here we pretrain the foundation model — a Llama causal decoder — on those sequences by next-token prediction, using Ray Train to run a plain-PyTorch training loop across the cluster.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd
import logging
import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Pretrain by predicting the next token

We pretrain the model the way a language model learns text: **next-token prediction** over each card's flat token stream. Given the history so far, the decoder predicts the next token; every position contributes a gradient, so one packed sequence yields hundreds of self-supervised predictions. No labels are involved — the transactions supervise themselves, which is what lets a foundation model learn from far more data than the scarce fraud labels alone.

The model is a **Llama causal decoder** (`src/model.py`, built from `transformers` — GQA, SwiGLU, RMSNorm, RoPE), sized per scale (`full` ≈ 29M params, matching NVIDIA's blueprint). "Causal" means each position attends only to earlier ones, so predicting the next token can't cheat by peeking ahead — and the final position's hidden state, which has seen the whole history, becomes the customer embedding in Part 5.

## The training loop is plain PyTorch — Ray Train scales it

The function that does the work (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, forward, backward, step — with an AdamW optimizer on a **warmup + cosine-decay** learning-rate schedule (the schedule and betas/weight-decay come from the scale config). Ray Train wraps it with the distributed machinery you'd otherwise hand-write — it launches the workers, shards the dataset across them (`get_dataset_shard` + `iter_torch_batches`), wraps the model in DDP (`prepare_model`), and persists checkpoints to shared storage.

The payoff is the `ScalingConfig`: moving from one CPU worker here to many GPU workers at `full` is a one-line change (`num_workers`, `use_gpu`), not a rewrite of the loop. `scripts/03_pretrain.py` wraps this same `train_func` for headless runs, so the walkthrough and the job train identically.

Note: ~12 min on `small` (1×L40S). At `full` (40 epochs, 8 GPU workers, 4096-token sequences) budget a couple of hours.

In [2]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # train_func is shared with scripts/03_pretrain.py
import math

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]
train_ds = ray.data.read_parquet(paths["tokenized_pretrain"])

# The cosine LR schedule steps once per optimizer step, so it needs the total
# step count up front: each worker steps once per batch of its shard, i.e.
# (sequences / workers / batch) * epochs. Count the sequences once here.
n_seqs = train_ds.count()
steps_per_epoch = max(1, math.ceil(n_seqs / pt["num_workers"] / pt["batch_size"]))
total_steps = steps_per_epoch * pt["epochs"]
warmup_steps = int(pt.get("warmup_ratio", 0.0) * total_steps)
print(f"pretrain (causal LM): {n_seqs:,} sequences · global batch {pt['batch_size'] * pt['num_workers']} "
      f"· ~{steps_per_epoch} steps/epoch × {pt['epochs']} = ~{total_steps:,} optimizer steps "
      f"(warmup {warmup_steps})")

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": paths["vocab"], "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        # AdamW betas/weight-decay + warmup/cosine LR schedule (from the scale config).
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    # The one line that moves laptop -> cluster: number of workers, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name="transaction_fm_pretrain",
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy weights to the canonical path for Part 5
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

2026-07-02 23:02:31,445	INFO logging.py:416 -- Registered dataset logger for dataset dataset_632_0
2026-07-02 23:02:31,477	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_632_0. Full logs are in /tmp/ray/session_2026-07-02_10-58-50_130263_2811/logs/ray-data
2026-07-02 23:02:31,477	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_632_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[MapBatches(count_rows)]
2026-07-02 23:02:31,483	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 28.5% of available memory (583.2GiB out of 2048.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-07-02 23:02:31,485	INFO __init__.py:56 -- Progress will be log

pretrain (causal LM): 64,561 sequences · global batch 64 · ~1009 steps/epoch × 8 = ~8,072 optimizer steps (warmup 403)


(TrainController pid=372957) Requesting resources: {'GPU': 1} * 8
(TrainController pid=372957) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=372957) Attempting to start training worker group of size 8 with the following resources: [{'GPU': 1}] * 8
(RayTrainWorker pid=47654, ip=10.0.122.23) Setting up process group for: env:// [rank=0, world_size=8]
(RayTrainWorker pid=43582, ip=10.0.85.107) Moving model to device: cuda:0
(RayTrainWorker pid=42799, ip=10.0.64.211) Wrapping provided model in DistributedDataParallel.
(TrainController pid=372957) Started training worker group of size 8: 
(TrainController pid=372957) - (ip=10.0.122.23, pid=47654) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=372957) - (ip=10.0.64.211, pid=42799) world_rank=1, local_rank=0, node_rank=1
(TrainController pid=372957) - (ip=10.0.71.101, pid=46180) world_rank=2, local_rank=0, node_rank=2
(TrainController pid=372957) - (ip=10.0.121.86, pid=43404) world_rank=3, local_rank=0, nod

(pid=373225) Running Dataset train_633_0.: 0.00 row [00:00, ? row/s]

(pid=373225) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=373225) - ReadFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=373225) - split(8, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=373225) Registered dataset logger for dataset train_633_0
(SplitCoordinator pid=373225) Starting execution of Dataset train_633_0. Full logs are in /tmp/ray/session_2026-07-02_10-58-50_130263_2811/logs/ray-data
(SplitCoordinator pid=373225) Execution plan of Dataset train_633_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> OutputSplitter[split(8, equal=True)]
(SplitCoordinator pid=373225) ⚠️  Ray's object store is configured to use only 28.5% of available memory (583.2GiB out of 2048.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=373225) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bar

WorkerGroupError: Training failed due to worker errors:
[Rank 0,1,2,3,4,5,6,7 Error Snippet]:
Traceback (most recent call last):
  File "/tmp/ray/session_2026-07-02_10-58-50_130263_2811/runtime_resources/working_dir_files/_ray_pkg_79f59a890baad34a/src/pretrain.py", line 89, in train_func
    loss, _ = model({"input_ids": input_ids, "attention_mask": attn, "labels": labels})
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ray/session_2026-07-02_10-58-50_130263_2811/runtime_resources/pip/42af90ee71173d51487ef71cff820f496665a6ab/virtualen...
... (Output truncated. See individual worker logs for full details) ...
56 GiB of which 168.81 MiB is free. Including non-PyTorch memory, this process has 14.39 GiB memory in use. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 10.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to a

## Reading the metrics

Watch **causal-LM loss** (cross-entropy) and its exponent, **perplexity** — the model's effective "how many tokens am I choosing between." Perplexity well below the vocabulary size (~6.3k) means the decoder has learned real next-token structure rather than guessing uniformly; it should fall epoch over epoch and settle as the cosine LR decays.

At `mini` (2 CPU epochs, a tiny 2-layer model) this is only a smoke test — the number that matters is the downstream fraud lift in Part 6, from the `small`/`full` GPU pretrain.

In [ ]:
from src.tokenizer import VOCAB_SIZE

m = result.metrics
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")
print(f"vocab {VOCAB_SIZE:,} tokens — perplexity well below vocab size means the decoder "
      "learned real next-token structure (not uniform guessing).")

## Takeaways

**Ray Train**
- The training loop is plain PyTorch; Ray Train adds the distributed parts — worker launch, dataset sharding, DDP/FSDP, checkpointing, fault tolerance — without changing the loop.
- `ScalingConfig` is the one knob that moves the same code from one CPU worker here to N GPU workers at `full` (`num_workers`, `use_gpu`); the model fits one GPU at every scale (data-parallel DDP), with `use_fsdp` available if it ever outgrows one.
- Checkpoints persist to shared storage (`storage_path`), so workers on other nodes can write them and downstream stages can read them.
- The notebook runs the same `train_func` that `scripts/03_pretrain.py` runs headless.

**The model**
- Self-supervised **next-token prediction** over the flat token stream — a Llama causal decoder, no labels required.
- Built from `transformers` (`LlamaConfig`/`LlamaForCausalLM`) at NVIDIA's blueprint config, so the architecture is faithful rather than hand-rolled.
- The trained decoder's last-token hidden state becomes the customer embedding in Part 5.

---

## Next

**Part 5 — Batch embedding extraction**: run the pretrained decoder over every card with Ray Data — a heterogeneous CPU-read + GPU-infer pipeline that writes one embedding per window.